# Experiment 2 - LunarLanderContinuous-v3

## Experiment 2
**Task**: Solve "LunarLanderContinuous-v3".

### Objectives:
1.  **Implement an On-Policy Agent** (PPO)
2.  **Implement an Off-Policy Agent** (SAC)
3.  **Hyperparameter Tuning**: Perform Grid Search to find stable agents.
4.  **Comparative Analysis**: Learning curves (reward vs timesteps) of the best agents.


In [ ]:
!pip install gymnasium[box2d] swig renderlab moviepy

In [ ]:
import gymnasium as gym
from gymnasium.vector import AsyncVectorEnv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
from torch.optim import AdamW
import matplotlib.pyplot as plt
from collections import namedtuple, deque
import random
import copy
import itertools
from itertools import count
import warnings
warnings.filterwarnings("ignore")

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. On-Policy Agent: PPO-GAE
Adapted from examples to solve `LunarLanderContinuous-v3`.

In [ ]:
class ActorNet(nn.Module):
    def __init__(self, input_size, hidden_units, output_size):
        super(ActorNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_units),
            nn.Tanh(),
            nn.Linear(hidden_units, int(hidden_units/2)),
            nn.Tanh()
        )
        self.mu_head = nn.Linear(int(hidden_units/2), output_size)
        self.logstd_head = nn.Linear(int(hidden_units/2), output_size)

    def forward(self, x):
        x = self.model(x)
        # LunarLanderContinuous action range is [-1, 1]
        loc = torch.tanh(self.mu_head(x)) 
        scale = torch.exp(self.logstd_head(x))
        return loc, scale

class CriticNet(nn.Module):
    def __init__(self, input_size, hidden_units):
        super(CriticNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_units),
            nn.Tanh(),
            nn.Linear(hidden_units, int(hidden_units/2)),
            nn.Tanh()
        )
        self.value_head = nn.Linear(int(hidden_units/2), 1)

    def forward(self, x):
        x = self.model(x)
        value = self.value_head(x)
        return value

In [ ]:
class RunningMem:
    def __init__(self):
        self.reset()

    def store(self, obs, action, logprob, reward, done, values):
        self.obs.append(obs)
        self.actions.append(action)
        self.logprobs.append(logprob)
        self.rewards.append(reward)
        self.dones.append(done)
        self.values.append(values)

    def reset(self):
        self.obs = []
        self.actions = []
        self.logprobs = []
        self.rewards = []
        self.dones = []
        self.values = []

In [ ]:
def train_ppo(params, name="PPO", seed=42, max_steps=500000):
    env_name = "LunarLanderContinuous-v3"
    def make_env():
        return gym.make(env_name)
    
    envs = AsyncVectorEnv([make_env for _ in range(params['N_ENVS'])])
    
    obs_dim = envs.single_observation_space.shape[0]
    act_dim = envs.single_action_space.shape[0]

    actor = ActorNet(obs_dim, params['HIDDEN_SIZE'], act_dim).to(device)
    critic = CriticNet(obs_dim, params['HIDDEN_SIZE']).to(device)
    
    opt_actor = AdamW(actor.parameters(), lr=params['LR'])
    opt_critic = AdamW(critic.parameters(), lr=params['LR'])

    obs, _ = envs.reset(seed=seed)
    mem = RunningMem()
    
    ep_rewards = deque(maxlen=params['N_ENVS']*5)
    running_rewards = np.zeros(params['N_ENVS'])
    
    tracker = {'steps': [], 'avg_reward': []}
    step_count = 0
    
    print(f"[{name}] Training PPO... LR={params['LR']} | BS={params['BATCH_SIZE']}")

    while step_count < max_steps:
        # 1. Collect Data
        for _ in range(params['N_STEPS']):
            step_count += params['N_ENVS']
            
            obs_t = torch.as_tensor(obs, dtype=torch.float32).to(device)
            with torch.no_grad():
                loc, scale = actor(obs_t)
                dist = Normal(loc, scale + 1e-6)
                action = dist.sample()
                log_prob = dist.log_prob(action).sum(axis=-1)
                value = critic(obs_t).squeeze()

            next_obs, rewards, terminations, truncations, _ = envs.step(action.cpu().numpy())
            dones = terminations | truncations

            mem.store(obs_t, action, log_prob, 
                      torch.as_tensor(rewards, dtype=torch.float32).to(device),
                      torch.as_tensor(dones, dtype=torch.float32).to(device),
                      value)
            
            running_rewards += rewards
            for i, done in enumerate(dones):
                if done:
                    ep_rewards.append(running_rewards[i])
                    running_rewards[i] = 0

            obs = next_obs

        # 2. Compute Advantages (GAE)
        with torch.no_grad():
            next_value = critic(torch.as_tensor(next_obs, dtype=torch.float32).to(device)).squeeze()
        
        # Flatten buffer
        b_obs = torch.stack(mem.obs)
        b_actions = torch.stack(mem.actions)
        b_logprobs = torch.stack(mem.logprobs)
        b_rewards = torch.stack(mem.rewards)
        b_dones = torch.stack(mem.dones)
        b_values = torch.stack(mem.values)

        advantages = torch.zeros_like(b_rewards).to(device)
        lastgaelam = 0
        for t in reversed(range(params['N_STEPS'])):
            if t == params['N_STEPS'] - 1:
                nextnonterminal = 1.0 - b_dones[t]
                nextvalues = next_value
            else:
                nextnonterminal = 1.0 - b_dones[t]
                nextvalues = b_values[t+1]
            
            delta = b_rewards[t] + params['GAMMA'] * nextvalues * nextnonterminal - b_values[t]
            advantages[t] = lastgaelam = delta + params['GAMMA'] * params['GAE_LAMBDA'] * nextnonterminal * lastgaelam

        returns = advantages + b_values

        # Flatten for update
        b_obs = b_obs.view(-1, obs_dim)
        b_actions = b_actions.view(-1, act_dim)
        b_logprobs = b_logprobs.view(-1)
        b_returns = returns.view(-1)
        b_advantages = advantages.view(-1)

        # 3. Optimize Policy & Value
        b_inds = np.arange(params['N_ENVS'] * params['N_STEPS'])
        for _ in range(params['N_EPOCHS']):
            np.random.shuffle(b_inds)
            for start in range(0, len(b_inds), params['BATCH_SIZE']):
                end = start + params['BATCH_SIZE']
                mb_inds = b_inds[start:end]

                new_loc, new_scale = actor(b_obs[mb_inds])
                new_dist = Normal(new_loc, new_scale + 1e-6)
                new_logprob = new_dist.log_prob(b_actions[mb_inds]).sum(axis=-1)
                
                ratio = torch.exp(new_logprob - b_logprobs[mb_inds])
                mb_advantages = b_advantages[mb_inds]
                mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)

                surr1 = ratio * mb_advantages
                surr2 = torch.clamp(ratio, 1.0 - params['CLIP_EPS'], 1.0 + params['CLIP_EPS']) * mb_advantages
                actor_loss = -torch.min(surr1, surr2).mean()
                
                new_value = critic(b_obs[mb_inds]).squeeze()
                critic_loss = F.mse_loss(new_value, b_returns[mb_inds])

                opt_actor.zero_grad()
                actor_loss.backward()
                opt_actor.step()

                opt_critic.zero_grad()
                critic_loss.backward()
                opt_critic.step()

        mem.reset()

        # Logging
        if len(ep_rewards) > 0:
            avg_r = np.mean(ep_rewards)
            tracker['steps'].append(step_count)
            tracker['avg_reward'].append(avg_r)
            
            if step_count % 50000 < params['N_ENVS'] * params['N_STEPS']:
                print(f"{name} Step: {step_count} | Avg Reward: {avg_r:.2f}")
            
            if avg_r >= 195.0:
                print(f"{name} Solved at Step {step_count} with Avg Reward {avg_r:.2f}!")
                break
                
    envs.close()
    return tracker, actor

## 2. Off-Policy Agent: SAC
Adapted from `Examples/SAC (Pendulum).ipynb` for `LunarLanderContinuous-v3`.

In [ ]:
class SACPolicyNet(nn.Module):
    def __init__(self, input_size, hidden_units, output_size):
        super(SACPolicyNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, int(hidden_units/2)),
            nn.ReLU()
        )
        self.mu_head = nn.Linear(int(hidden_units/2), output_size)
        self.logstd_head = nn.Linear(int(hidden_units/2), output_size)

    def forward(self, x):
        x = self.model(x)
        mu = self.mu_head(x)
        log_std = self.logstd_head(x)
        log_std = torch.clamp(log_std, -20, 2)
        std = torch.exp(log_std)
        
        dist = Normal(mu, std)
        u = dist.rsample()
        action = torch.tanh(u) # Range [-1, 1]
        
        log_prob = dist.log_prob(u)
        log_prob -= 2 * (np.log(2) - u - F.softplus(-2*u))
        log_prob = log_prob.sum(dim=-1, keepdim=True)
        
        return action, log_prob

class SACQNet(nn.Module):
    def __init__(self, input_size, hidden_units):
        super(SACQNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, int(hidden_units/2)),
            nn.ReLU(),
            nn.Linear(int(hidden_units/2), 1)
        )

    def forward(self, state, action):
        x = torch.cat([state, action], 1)
        return self.model(x)

In [ ]:
Transition = namedtuple('Transition', ('states', 'actions', 'rewards', 'dones', 'next_states'))

class ReplayBuffer:
    def __init__(self, capacity, batch_size):
        self.memory = deque(maxlen=capacity)
        self.batch_size = batch_size

    def append(self, transition):
        self.memory.append(transition)

    def sample(self):
        batch = random.sample(self.memory, self.batch_size)
        batch = Transition(*zip(*batch))
        return (
            torch.tensor(np.array(batch.states), dtype=torch.float32).to(device),
            torch.tensor(np.array(batch.actions), dtype=torch.float32).to(device),
            torch.tensor(np.array(batch.rewards), dtype=torch.float32).unsqueeze(1).to(device),
            torch.tensor(np.array(batch.dones), dtype=torch.float32).unsqueeze(1).to(device),
            torch.tensor(np.array(batch.next_states), dtype=torch.float32).to(device)
        )

In [ ]:
def train_sac(params, name="SAC", seed=42, max_steps=300000):
    env = gym.make("LunarLanderContinuous-v3")
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]

    # Networks
    actor = SACPolicyNet(obs_dim, params['HIDDEN_SIZE'], act_dim).to(device)
    q1 = SACQNet(obs_dim + act_dim, params['HIDDEN_SIZE']).to(device)
    q2 = SACQNet(obs_dim + act_dim, params['HIDDEN_SIZE']).to(device)
    
    # Targets
    q1_target = copy.deepcopy(q1)
    q2_target = copy.deepcopy(q2)

    # Optimizers
    actor_opt = AdamW(actor.parameters(), lr=params['LR'])
    q_opt = AdamW(list(q1.parameters()) + list(q2.parameters()), lr=params['LR'])

    memory = ReplayBuffer(params['BUFFER_SIZE'], params['BATCH_SIZE'])
    tracker = {'steps': [], 'avg_reward': []}
    
    obs, _ = env.reset(seed=seed)
    step_count = 0
    episode_reward = 0
    recent_scores = deque(maxlen=50)
    
    print(f"[{name}] Training SAC... LR={params['LR']} | BS={params['BATCH_SIZE']}")

    while step_count < max_steps:
        step_count += 1
        
        # Action Selection
        if step_count < params['MIN_REPLAY_SIZE']:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                state_t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
                action, _ = actor(state_t)
                action = action.cpu().numpy()[0]

        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        memory.append(Transition(obs, action, reward, terminated, next_obs))
        obs = next_obs
        episode_reward += reward

        if done:
            recent_scores.append(episode_reward)
            avg_score = np.mean(recent_scores)
            tracker['steps'].append(step_count)
            tracker['avg_reward'].append(avg_score)
            
            if len(tracker['steps']) % 50 == 0:
                print(f"{name} Step: {step_count} | Avg Reward: {avg_score:.2f}")
            
            if avg_score >= 195.0:
                print(f"{name} Solved at Step {step_count} with Avg Reward {avg_score:.2f}!")
                break

            obs, _ = env.reset()
            episode_reward = 0

        # Training
        if step_count >= params['MIN_REPLAY_SIZE']:
            b_states, b_actions, b_rewards, b_dones, b_next_states = memory.sample()
            
            # Critic Update
            with torch.no_grad():
                next_actions, next_log_probs = actor(b_next_states)
                q1_next = q1_target(b_next_states, next_actions)
                q2_next = q2_target(b_next_states, next_actions)
                min_q_next = torch.min(q1_next, q2_next)
                target_q = b_rewards + params['GAMMA'] * (1 - b_dones) * (min_q_next - params['ALPHA'] * next_log_probs)

            current_q1 = q1(b_states, b_actions)
            current_q2 = q2(b_states, b_actions)
            q_loss = F.mse_loss(current_q1, target_q) + F.mse_loss(current_q2, target_q)

            q_opt.zero_grad()
            q_loss.backward()
            q_opt.step()

            # Actor Update
            new_actions, log_probs = actor(b_states)
            q1_pi = q1(b_states, new_actions)
            q2_pi = q2(b_states, new_actions)
            min_q_pi = torch.min(q1_pi, q2_pi)
            actor_loss = (params['ALPHA'] * log_probs - min_q_pi).mean()

            actor_opt.zero_grad()
            actor_loss.backward()
            actor_opt.step()

            # Soft Update
            for target_param, param in zip(q1_target.parameters(), q1.parameters()):
                target_param.data.copy_(param.data * params['TAU'] + target_param.data * (1.0 - params['TAU']))
            for target_param, param in zip(q2_target.parameters(), q2.parameters()):
                target_param.data.copy_(param.data * params['TAU'] + target_param.data * (1.0 - params['TAU']))

    return tracker, actor

## 3. Hyperparameter Tuning (Grid Search)

In [ ]:
# Grid Search for PPO
PPO_LRS = [3e-4, 1e-3]
PPO_BATCHES = [64, 128]

ppo_grid_results = {}
best_ppo_reward = -float('inf')
best_ppo_model = None
best_ppo_results = None

print("Starting PPO Grid Search...")
for lr in PPO_LRS:
    for bs in PPO_BATCHES:
        params = {
            'LR': lr,
            'BATCH_SIZE': bs,
            'GAMMA': 0.99,
            'GAE_LAMBDA': 0.95,
            'CLIP_EPS': 0.2,
            'N_EPOCHS': 5,
            'N_ENVS': 8,
            'N_STEPS': 512,
            'HIDDEN_SIZE': 128
        }
        run_name = f"PPO_LR{lr}_BS{bs}"
        results, model = train_ppo(params, name=run_name, max_steps=200000) # Short runs for tuning
        
        ppo_grid_results[run_name] = results
        final_avg = results['avg_reward'][-1] if len(results['avg_reward']) > 0 else -999
        
        if final_avg > best_ppo_reward:
            best_ppo_reward = final_avg
            best_ppo_model = model
            best_ppo_results = results
            print(f"New Best PPO: {run_name} (Avg: {final_avg:.2f})")

# Grid Search for SAC
SAC_LRS = [3e-4, 1e-3]
SAC_BATCHES = [128, 256]

sac_grid_results = {}
best_sac_reward = -float('inf')
best_sac_model = None
best_sac_results = None

print("\nStarting SAC Grid Search...")
for lr in SAC_LRS:
    for bs in SAC_BATCHES:
        params = {
            'LR': lr,
            'BATCH_SIZE': bs,
            'GAMMA': 0.99,
            'BUFFER_SIZE': 50000,
            'MIN_REPLAY_SIZE': 1000,
            'TAU': 0.005,
            'ALPHA': 0.2,
            'HIDDEN_SIZE': 256
        }
        run_name = f"SAC_LR{lr}_BS{bs}"
        results, model = train_sac(params, name=run_name, max_steps=100000) # SAC learns faster in steps terms
        
        sac_grid_results[run_name] = results
        final_avg = results['avg_reward'][-1] if len(results['avg_reward']) > 0 else -999
        
        if final_avg > best_sac_reward:
            best_sac_reward = final_avg
            best_sac_model = model
            best_sac_results = results
            print(f"New Best SAC: {run_name} (Avg: {final_avg:.2f})")

## 4. Final Comparison & Visualization

In [ ]:
plt.figure(figsize=(12, 8))

# Plot PPO runs
for name, res in ppo_grid_results.items():
    plt.plot(res['steps'], res['avg_reward'], alpha=0.3, label=name)
    
# Plot SAC runs
for name, res in sac_grid_results.items():
    plt.plot(res['steps'], res['avg_reward'], alpha=0.3, linestyle='--', label=name)

# Highlight best
plt.plot(best_ppo_results['steps'], best_ppo_results['avg_reward'], linewidth=3, color='blue', label='Best PPO')
plt.plot(best_sac_results['steps'], best_sac_results['avg_reward'], linewidth=3, color='orange', label='Best SAC')

plt.xlabel('Environment Timesteps')
plt.ylabel('Average Reward (50-ep)')
plt.title('PPO vs SAC Hyperparameter Tuning & Comparison')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import renderlab as rl

def view_agent(model, name, is_stochastic=True):
    print(f"Visualizing {name}...")
    env = gym.make("LunarLanderContinuous-v3", render_mode="rgb_array")
    env = rl.RenderFrame(env, f"./video_{name}")
    
    obs, _ = env.reset()
    
    while True:
        with torch.no_grad():
            obs_t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
            if is_stochastic:
                action, _ = model(obs_t)
                if isinstance(action, tuple): 
                    action = action[0]
            else:
                action = model(obs_t)
                
            action = action.cpu().numpy()[0]
            
        obs, reward, terminated, truncated, _ = env.step(action)
        if terminated or truncated:
            break
    
    env.play()

if best_ppo_model:
    view_agent(best_ppo_model, "Best_PPO")

if best_sac_model:
    view_agent(best_sac_model, "Best_SAC")